# 02 Analysis

I use this notebook to reproduce the main analysis behind the interactive report in `docs/index.html`. I start from the processed CSV files created in `01_data_cleaning.ipynb`, then calculate the headline checks and build the figures used to interpret the spatial pattern of conflict events.





## Step 0: Load processed data and summary files

I set up the project folder and load the processed data files created in `01_data_cleaning.ipynb`. I read the event-level processed CSV and the three summary CSV files, then keep the 1997-2024 report sample so the analysis matches the final report. I render Plotly figures inside the notebook output cells using HTML display, so the figures can be inspected directly without depending on Plotly's optional nbformat renderer.





In [ ]:
from functools import lru_cache
import json
from pathlib import Path
import subprocess
import sys
from typing import Dict

from IPython.display import HTML, IFrame, display

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

REPO_URL = "https://github.com/lexieliujy/POLI3148_PS1.git"


def show_plotly_figure(fig: go.Figure) -> None:
    display(
        HTML(
            fig.to_html(
                include_plotlyjs="cdn",
                full_html=False,
                config={"displayModeBar": False, "responsive": True},
            )
        )
    )


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "docs" / "index.html").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root folder.")


try:
    ROOT_DIR = find_project_root(Path.cwd())
except FileNotFoundError:
    ROOT_DIR = Path.cwd() / "POLI3148_PS1"
    if not (ROOT_DIR / "data").exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT_DIR)], check=True)

CODE_DIR = ROOT_DIR / "code"
DATA_DIR = ROOT_DIR / "data"
BOUNDARY_DATA_PATH = DATA_DIR / "support" / "ne_50m_admin_0_countries.zip"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

processed = pd.read_csv(DATA_DIR / "acled_mena_processed.csv", parse_dates=["event_date"])
yearly = pd.read_csv(DATA_DIR / "yearly_shift_summary.csv")
spatial = pd.read_csv(DATA_DIR / "spatial_bucket_summary.csv")
country = pd.read_csv(DATA_DIR / "country_pattern_summary.csv")

if processed.empty:
    raise RuntimeError("Processed data loaded as empty; check the data files.")



report_df = processed[processed["year"].between(1997, 2024)].copy()
report_df.shape








## Step 1: Define analysis helper functions

I define the reusable helper functions and shared settings used by the analysis figures. The full event-level map code is written directly in Step 7 so the map construction is visible in one place.






### 1.1 Set shared colors, capital names, and map boundaries

I define a small color palette so the figures use the same visual language throughout the notebook. I also define the capital-city lookup used by the map helper, and I create `load_world_geojson()` to load country boundaries only when the map needs them.





In [ ]:
PALETTE = {
    "capital_remote": "#c44536",
    "border_battles": "#1f4e79",
    "remote_total": "#d98b5f",
    "other": "#6c7a89",
    "civilians": "#d8a47f",
}

CAPITALS_BY_COUNTRY: Dict[str, str] = {
    "Algeria": "Algiers",
    "Bahrain": "Manama",
    "Egypt": "Cairo",
    "Iran": "Tehran",
    "Iraq": "Baghdad",
    "Israel": "Jerusalem",
    "Jordan": "Amman",
    "Kuwait": "Kuwait City",
    "Lebanon": "Beirut",
    "Libya": "Tripoli",
    "Morocco": "Rabat",
    "Oman": "Muscat",
    "Saudi Arabia": "Riyadh",
    "Sudan": "Khartoum",
    "Syria": "Damascus",
    "Turkey": "Ankara",
    "Tunisia": "Tunis",
    "United Arab Emirates": "Abu Dhabi",
    "Yemen": "Sanaa",
}


@lru_cache(maxsize=1)
def load_world_geojson() -> dict:
    try:
        import geopandas as gpd
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "The optional map requires geopandas. Install the notebook dependencies with "
            "`pip install -r requirements.txt` in the same Python environment used by this notebook."
        ) from exc

    world = gpd.read_file(f"zip://{BOUNDARY_DATA_PATH}")[["ADMIN", "geometry"]].copy()
    world = world.rename(columns={"ADMIN": "country_name"})
    return json.loads(world.to_json())





### 1.2 Define reusable chart builders

I also define reusable chart functions for the main time-series and composition plots. These functions turn the summary tables into Plotly figures without creating new variables or altering the cleaned data.





In [ ]:
def build_trend_figure(yearly: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=yearly["year"],
            y=yearly["border_battle_events"],
            mode="lines+markers",
            name="Border-proximate battles",
            line={"color": PALETTE["border_battles"], "width": 3},
            marker={"size": 6},
            hovertemplate="Year %{x}<br>Events %{y}<extra></extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=yearly["year"],
            y=yearly["capital_remote_events"],
            mode="lines+markers",
            name="Capital-area remote attacks",
            line={"color": PALETTE["capital_remote"], "width": 3},
            marker={"size": 6},
            hovertemplate="Year %{x}<br>Events %{y}<extra></extra>",
        )
    )
    fig.update_layout(
        title="Border-proximate battles remain more common than capital-area remote attacks",
        template="plotly_white",
        height=430,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
        margin={"l": 40, "r": 20, "t": 70, "b": 45},
        xaxis_title="Year",
        yaxis_title="Recorded events",
    )
    return fig


def build_remote_growth_figure(yearly: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=yearly["year"],
            y=yearly["drone_events"],
            name="Air/drone strikes",
            marker_color=PALETTE["capital_remote"],
            opacity=0.82,
            hovertemplate="Year %{x}<br>Drone events %{y}<extra></extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=yearly["year"],
            y=yearly["all_remote_events"],
            mode="lines+markers",
            name="All remote violence events",
            line={"color": PALETTE["border_battles"], "width": 3},
            marker={"size": 6},
            hovertemplate="Year %{x}<br>Remote events %{y}<extra></extra>",
        )
    )
    fig.update_layout(
        title="Drone strikes and remote violence both rise sharply over time in MENA",
        template="plotly_white",
        height=430,
        margin={"l": 40, "r": 20, "t": 70, "b": 45},
        xaxis_title="Year",
        yaxis_title="Recorded events",
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    )
    return fig


def build_share_figure(yearly: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=yearly["year"],
            y=yearly["border_share_of_battles_pct"],
            name="Border-proximate share of all battles",
            mode="lines+markers",
            line={"color": PALETTE["border_battles"], "width": 3},
            marker={"size": 7},
            hovertemplate="Year %{x}<br>%{y}% of battles<extra></extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=yearly["year"],
            y=yearly["capital_share_of_remote_pct"],
            mode="lines+markers",
            name="Capital-area share of all remote attacks",
            line={"color": PALETTE["capital_remote"], "width": 3},
            marker={"size": 7},
            hovertemplate="Year %{x}<br>%{y}% of remote events<extra></extra>",
        )
    )
    fig.update_layout(
        title="Neither share shows a clean shift from border conflict to capital targeting",
        template="plotly_white",
        height=430,
        margin={"l": 40, "r": 20, "t": 70, "b": 45},
        xaxis_title="Year",
        yaxis={"title": "Share (%)"},
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    )
    return fig


def build_spatial_composition_figure(df: pd.DataFrame) -> go.Figure:
    composition = (
        pd.crosstab(df["spatial_bucket"], df["event_type"], normalize="index")
        .mul(100)
        .reset_index()
        .melt(id_vars="spatial_bucket", var_name="event_type", value_name="share_pct")
    )
    fig = px.bar(
        composition,
        x="spatial_bucket",
        y="share_pct",
        color="event_type",
        barmode="stack",
        category_orders={
            "spatial_bucket": ["Capital areas", "Border-proximate areas", "Other areas"]
        },
        color_discrete_map={
            "Battles": PALETTE["border_battles"],
            "Explosions/Remote violence": PALETTE["capital_remote"],
            "Violence against civilians": PALETTE["civilians"],
        },
        labels={
            "spatial_bucket": "",
            "share_pct": "Share of events (%)",
            "event_type": "Event type",
        },
        title="Capital areas are more remote than the rest of the region, but not dominant",
    )
    fig.update_layout(
        template="plotly_white",
        height=420,
        margin={"l": 40, "r": 20, "t": 70, "b": 45},
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    )
    fig.update_traces(hovertemplate="%{x}<br>%{fullData.name}: %{y:.1f}%<extra></extra>")
    return fig





## Step 2: State the research question

I got the inspiration for the research question because of the lesson (POLI3164: International Relations of the Middle East and North Africa). The conflicts in MENA region has been specifically frequent in recent years, and when going through the example ACLED analysis report, I noticed that the trend of remote attacks, especially those using drones has been increased sharply. I was wondering if there is any connection between the intensity of conflicts in MENA and how the use of remote attacks may potentially contributed to that and also the contemporary strategic mechanism (although after a series of data analysis, it seems to rebut my assumption, I think it is still of some analytical value). 

Therefore I frame my research qeustion as whether the rise of drones and other remote attacks in the Middle East and North Africa has been accompanied by a spatial shift from border-proximate conflict toward attacks on capital areas.

It should be noted that the analysis is descriptive rather than causal. 





### 2.1 Calculate headline counts

I calculate the headline counts used to summarize the sample before making the figures. I count all events, border-proximate events, capital-area events, and capital-area remote events so the reader can see the size of each category.





In [ ]:
headline = {
    "all_events": len(report_df),
    "border_proximate_events": int(report_df["is_border_proximate"].sum()),
    "border_proximate_share_pct": round(report_df["is_border_proximate"].mean() * 100, 2),
    "capital_area_events": int(report_df["is_capital_area"].sum()),
    "capital_area_share_pct": round(report_df["is_capital_area"].mean() * 100, 2),
    "capital_remote_events": int(report_df["capital_remote_event"].sum()),
    "capital_remote_share_of_remote_pct": round(
        report_df["capital_remote_event"].sum() / report_df["is_remote"].sum() * 100, 2
    ),
}

headline





## Step 3: Build Figure 1, remote violence and drone strikes over time

I build Figure 1 to show why the research question starts from remote violence and drones. I use a line for all explosions/remote violence events and bars for air/drone strike events so the reader can compare the overall remote-attack trend with the drone-specific trend in the same figure.





In [ ]:
fig1 = build_remote_growth_figure(yearly)
show_plotly_figure(fig1)





## Step 4: Build Figure 2, capital and border shares within remote attacks

I build Figure 2 to compare two shares within remote violence events: the share that occurs near borders and the share that occurs in capital areas. I use percentages rather than raw counts so the trend is not driven only by the overall growth of remote violence over time.





In [ ]:
remote_yearly = (
    report_df.groupby("year")
    .agg(
        all_remote_events=("is_remote", "sum"),
        border_remote_events=("is_border_proximate", lambda x: int((x & report_df.loc[x.index, "is_remote"]).sum())),
        capital_remote_events=("capital_remote_event", "sum"),
    )
    .reset_index()
)
remote_yearly["border_remote_share_pct"] = (
    remote_yearly["border_remote_events"] / remote_yearly["all_remote_events"] * 100
).round(2)
remote_yearly["capital_remote_share_pct"] = (
    remote_yearly["capital_remote_events"] / remote_yearly["all_remote_events"] * 100
).round(2)

fig2 = go.Figure()
fig2.add_trace(
    go.Scatter(
        x=remote_yearly["year"],
        y=remote_yearly["border_remote_share_pct"],
        mode="lines+markers",
        name="Border-proximate share of remote attacks",
        line={"color": "#1f4e79", "width": 3.2},
    )
)
fig2.add_trace(
    go.Scatter(
        x=remote_yearly["year"],
        y=remote_yearly["capital_remote_share_pct"],
        mode="lines+markers",
        name="Capital-area share of remote attacks",
        line={"color": "#c44536", "width": 3.0},
    )
)
fig2.update_layout(
    title="Within remote attacks, border-proximate share remains above capital-area share",
    template="plotly_white",
    xaxis_title="Year",
    yaxis_title="Share (%)",
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
)
show_plotly_figure(fig2)





## Step 5: Build Figure 3, yearly spatial shares using all events as the denominator

I build Figure 3 to show how much of all recorded conflict activity falls into capital areas and non-capital border-proximate areas each year. I separate non-capital border-proximate events from capital-area events so the stacked bars do not double-count places that are both capital-area and near a border.





In [ ]:
yearly_geo = (
    report_df.assign(noncapital_border=report_df["is_border_proximate"] & ~report_df["is_capital_area"])
    .groupby("year")
    .agg(
        all_events=("event_id_cnty", "count"),
        noncapital_border_events=("noncapital_border", "sum"),
        capital_area_events=("is_capital_area", "sum"),
    )
    .reset_index()
)
yearly_geo["noncapital_border_share_pct"] = (
    yearly_geo["noncapital_border_events"] / yearly_geo["all_events"] * 100
).round(2)
yearly_geo["capital_area_share_pct"] = (
    yearly_geo["capital_area_events"] / yearly_geo["all_events"] * 100
).round(2)

fig3 = go.Figure()
fig3.add_trace(
    go.Bar(
        x=yearly_geo["year"],
        y=yearly_geo["noncapital_border_share_pct"],
        name="Non-capital border-proximate share",
        marker={"color": "#1f4e79"},
    )
)
fig3.add_trace(
    go.Bar(
        x=yearly_geo["year"],
        y=yearly_geo["capital_area_share_pct"],
        name="Capital-area share",
        marker={"color": "#c44536"},
    )
)
fig3.update_layout(
    title="Yearly all-event shares: border-proximate space remains heavier than capital space",
    template="plotly_white",
    barmode="stack",
    xaxis_title="Year",
    yaxis_title="Share of all events (%)",
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
)
show_plotly_figure(fig3)






## Step 6: Inspect summary tables

I display the spatial and country summary tables to connect the figures back to the aggregate values used in the report narrative. These tables make it easier to see which spatial buckets and countries carry the largest event counts.





In [ ]:
spatial





In [ ]:
country.head(10)





## Step 7: Build Figure 4, the event-level map

I build Figure 4 directly in this section so the full map-making process is visible. I first keep only capital-area and border-proximate events, then label each point by map category, scale marker size by fatalities, add Natural Earth country boundaries, add event points, and add capital-city markers.

I save the Plotly map as a standalone HTML file and embed it back into the notebook with an iframe. I do this because the map is too heavy to render reliably as a normal inline Plotly cell, but the iframe keeps the notebook figure visually consistent with the published interactive report.


In [ ]:
max_points_per_category = 18000
figure4_html_path = ROOT_DIR / "docs" / "figure4_notebook_map.html"

map_df = report_df[report_df["is_capital_area"] | report_df["is_border_proximate"]].copy()
map_df["map_category"] = "Border-proximate events"
map_df.loc[map_df["is_capital_area"], "map_category"] = "Capital-area events"
map_df["display_fatalities"] = map_df["fatalities"].clip(lower=0).fillna(0)
map_df["marker_size"] = (
    map_df["display_fatalities"].clip(upper=400).pow(0.5) * 1.6 + 6.2
).round(1)

capital_points = []
for country_name, capital in CAPITALS_BY_COUNTRY.items():
    capital_rows = report_df[
        (report_df["country"] == country_name)
        & (
            report_df["location"].astype(str).eq(capital)
            | report_df["location"].astype(str).str.startswith(f"{capital} -")
        )
    ][["latitude", "longitude"]].drop_duplicates()

    if len(capital_rows):
        point = capital_rows.iloc[0].to_dict()
        point["country"] = country_name
        point["capital"] = capital
        capital_points.append(point)

capitals_df = pd.DataFrame(capital_points)
world_geojson = load_world_geojson()

fig4 = go.Figure()
fig4.add_trace(
    go.Choroplethmap(
        geojson=world_geojson,
        locations=[feature["properties"]["country_name"] for feature in world_geojson["features"]],
        z=[1] * len(world_geojson["features"]),
        featureidkey="properties.country_name",
        colorscale=[[0, "#f7f3eb"], [1, "#f7f3eb"]],
        showscale=False,
        hoverinfo="skip",
        marker={"line": {"color": "#6b7280", "width": 1.0}},
        name="Country borders",
        showlegend=False,
    )
)

for category, color in [
    ("Capital-area events", PALETTE["capital_remote"]),
    ("Border-proximate events", PALETTE["border_battles"]),
]:
    sub = map_df[map_df["map_category"] == category].copy()

    if len(sub) > max_points_per_category:
        sub = sub.sort_values("event_date")
        step = len(sub) / max_points_per_category
        keep_idx = [int(i * step) for i in range(max_points_per_category)]
        sub = sub.iloc[keep_idx].copy()

    sub["event_date_str"] = sub["event_date"].dt.strftime("%Y-%m-%d")
    custom_cols = [
        "latitude",
        "longitude",
        "country",
        "event_date_str",
        "year",
        "event_type",
        "sub_event_type",
        "fatalities",
        "border_distance_km",
        "marker_size",
    ]

    fig4.add_trace(
        go.Scattermap(
            lat=sub["latitude"].tolist(),
            lon=sub["longitude"].tolist(),
            ids=sub["event_date_str"].tolist(),
            mode="markers",
            name=category,
            customdata=sub[custom_cols].values.tolist(),
            text=sub["location"].astype(str).tolist(),
            hovertemplate=(
                "%{text}<br>%{customdata[2]}<br>Date %{customdata[3]} (Year %{customdata[4]})"
                "<br>%{customdata[5]} | %{customdata[6]}"
                "<br>Fatalities %{customdata[7]}"
                "<br>Border distance %{customdata[8]:.1f} km<extra></extra>"
            ),
            marker={
                "size": sub["marker_size"].tolist(),
                "color": color,
                "opacity": 0.78,
            },
        )
    )

if not capitals_df.empty:
    fig4.add_trace(
        go.Scattermap(
            lat=capitals_df["latitude"].tolist(),
            lon=capitals_df["longitude"].tolist(),
            text=[""] * len(capitals_df),
            mode="markers",
            name="Capital cities",
            textposition="top center",
            hovertemplate="%{customdata[1]}<br>%{customdata[0]}<extra></extra>",
            customdata=capitals_df[["country", "capital"]].values.tolist(),
            marker={
                "size": 10,
                "symbol": "diamond",
                "color": "#111827",
            },
            textfont={"size": 10, "color": "#111827"},
        )
    )

fig4.update_layout(
    template="plotly_white",
    height=560,
    margin={"l": 20, "r": 20, "t": 70, "b": 20},
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    title="Spatial contrast between capital-area events and border-proximate events",
    map={
        "style": "white-bg",
        "center": {"lat": 31, "lon": 31},
        "zoom": 3.05,
        "bearing": 0,
        "pitch": 0,
    },
)

fig4.write_html(
    figure4_html_path,
    include_plotlyjs="cdn",
    full_html=True,
    config={"displayModeBar": False, "responsive": True, "scrollZoom": True},
)

print(
    f"Figure 4 map saved to {figure4_html_path.relative_to(ROOT_DIR)} with "
    f"{len(fig4.data)} Plotly traces."
)
figure4_iframe_src = figure4_html_path.relative_to(ROOT_DIR).as_posix()
print(f"Open the same map directly at {figure4_iframe_src} if the iframe does not render in your notebook viewer.")
display(IFrame(src=figure4_iframe_src, width="100%", height=620))



## Step 8: Conclusion

I use the four figures and summary tables to reach one stable descriptive conclusion: border-linked space remains the heavier conflict geography in MENA relative to capital space. The data do not support a simple regional story in which conflict moved from borders to capitals.




